<h3><b style="color:red">IMPORTANT:</b> Restart your session everytime you open this workshop</h3>

# Multimodal

In [1]:
# Load library
from transformers import pipeline

# Create a Image text to Text
A multimodal model that takes an image and a prompt as input and outputs a text response

## Use cases
* Visual Questin and Answer - Ask a question about an image
* Document understanding - Analyze charts, or get details from receipts

## Perform an inference from the created model

In [ ]:
image = "https://www.sheknows.com/wp-content/uploads/2018/08/FlickrTimDorrDogButtSniff_wu3qep.jpeg"
text = "The dogs are "
image = "https://static.boredpanda.com/blog/wp-content/uploads/2016/09/mother-bear-cubs-animal-parenting-21-57e3a2161d7f7__880.jpg"
text = "Where is the baby bear?"



In [2]:

# Import libraries
from transformers import AutoTokenizer, AutoModelForImageTextToText

In [3]:
# Load model, and processor
model_name = "Qwen/Qwen3-VL-2B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForImageTextToText.from_pretrained(model_name)


config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

## Chat templates

Chat templates are structured formatting schemes that define how conversations between users and language models are organized. They play a crucial role in ensuring the model correctly interprets the different roles and turns in a conversation.

LLMs are fundamentally trained on sequences of tokens. Without a template, the model would have no way to distinguish between what the user said, what the system prompt instructs. Chat templates solve this by injecting special tokens or formatting conventions that the model was trained to recognize.

Every chat-tuned model was fine-tuned on a specific text format. LLaMA expects something like `[INST] ... [/INST]`, ChatML uses `<|im_start|>user\n...<|im_end|>`, Qwen has its own variant, and so on. If you feed the model text in the wrong format, it still generates something, but quality collapses because the model never saw that pattern during training.

In [4]:
print(tokenizer.chat_template)


{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {%- if messages[0].content is string %}
            {{- messages[0].content }}
        {%- else %}
            {%- for content in messages[0].content %}
                {%- if 'text' in content %}
                    {{- content.text }}
                {%- endif %}
            {%- endfor %}
        {%- endif %}
        {{- '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if me

## Structured input

Create a structured input and convert it to the model's chat template



In [18]:
# Create structured input
image = "https://www.sheknows.com/wp-content/uploads/2018/08/FlickrTimDorrDogButtSniff_wu3qep.jpeg"
text = "The dogs are "
image = "https://static.boredpanda.com/blog/wp-content/uploads/2016/09/mother-bear-cubs-animal-parenting-21-57e3a2161d7f7__880.jpg"
text = "Where is the baby bear?"
image = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSd9fLsRwJ9acjOn_jIkoJK9D9hQTtO65Nc7lpfyclSWQyH5x9YyZVB7Q3d&s=10"
text = "What is the color of the animal?"

message = [
    {
        "role": "user",
        "content": [
            { "type": "image", "url": image },
            { "type": "text", "text": text}
        ]
    }
]



In [22]:
image = "https://static.boredpanda.com/blog/wp-content/uploads/2016/09/mother-bear-cubs-animal-parenting-21-57e3a2161d7f7__880.jpg"
text = "What is the colour of the bear?"
message = [
    {
        "role": "user",
        "content": [
            { "type": "image", "url": image },
            { "type": "text", "text": text}
        ]
    }
]

# Generate Prompt

Create the prompt with `apply_chat_template`.
* add_generation_prompt - append a special sequence of tokens to the end to instruct the model to generate response
* tokenize - tokenize the generated prompt
* return_dict - return the output as a dictionary vs a single tensor eg. `input_ids`, `attention_mask`, `pixel_values`
* return_tensors - pt (PyTorch), tf (TensorFlow)

In [23]:
prompt = tokenizer.apply_chat_template(
    message,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)
print(prompt)

{'input_ids': tensor([[151644,    872,    198, 151652, 151655, 151653,   3838,    374,    279,
          12463,    315,    279,  11722,     30, 151645,    198, 151644,  77091,
            198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [24]:
print(tokenizer.decode(prompt['input_ids'][0]))

<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>What is the colour of the bear?<|im_end|>
<|im_start|>assistant



In [11]:
print(tokenizer.decode(prompt['input_ids'][0]))

<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Where is the baby bear?<|im_end|>



## Dump the output
* input_ids - the tokens
* attention_mask - which token to attend, typically used to identify padding in batch
* mm_token_type_ids - identify the token type in the multimodal prompt
* pixel_values - the processed image data
* image_grid_thw - Time, Height, Width describes how the image are divided into patches or tiles for processing

In [15]:
for k, v in prompt.items():
    print(k, v.shape)

input_ids torch.Size([1, 17])
attention_mask torch.Size([1, 17])


# Inference

Generate response with the prompt from the model and decode the output

In [25]:
result = model.generate(**prompt)
print(result)

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=39) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


tensor([[151644,    872,    198, 151652, 151655, 151653,   3838,    374,    279,
          12463,    315,    279,  11722,     30, 151645,    198, 151644,  77091,
            198,    785,  11722,    304,    279,   2168,    374,  13876,     13,
         151645]])


In [26]:
print(tokenizer.decode(result[0]))

<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>What is the colour of the bear?<|im_end|>
<|im_start|>assistant
The bear in the image is brown.<|im_end|>
